<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Titans_jax_TPUv6e-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Gemma-Titans Training on TPU with Kauldron

This notebook demonstrates the complete pipeline for training and using the **Gemma-Titans** hybrid model on a Google Cloud TPU v5e using the `Kauldron` framework (DeepMind's standard training library).

### Steps included:
1. **Initialization:** Loading base Gemma weights and randomly initializing Titans NLTM modules using `SkipTitans`.
2. **Pre-training (CPT):** Training *only* the Titans memory layers using `kd.optim.partial_updates` to avoid TPU OOM, utilizing a proper dataset.
3. **Save/Load:** Splitting the PyTree to save only the fine-tuned memory weights.
4. **Dialogue Inference:** Running the model and updating the internal memory cache dynamically.

In [ ]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone --depth 1 https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone --depth 1 https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone --depth 1 https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio





In [ ]:
!git clone --depth 1 https://github.com/andrew-veriga/Titans_jax.git


In [ ]:
!pip install importlib_resources

# Start

In [ ]:
import os
os._exit(0)

In [ ]:
# !git pull

In [ ]:
# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())
from kauldron import kd
from kauldron.ktyping import Array

In [ ]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import os
import orbax.checkpoint as ocp
import dataclasses

# Gemma imports
from gemma import gm
from gemma.gm.nn import _config

# Our custom Titans integration
import importlib

%cd Titans_jax
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
""" старые настройки
# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"
"""
# Разрешаем JAX забрать память сразу для максимальной скорости
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"

# Увеличиваем долю памяти (оставляем чуть-чуть на системные нужды)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

# Оптимизируем флаги для производительности, а не для экономии
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true " # Скрывает задержки памяти за вычислениями
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728" # Оптимально для больших батчей
)

os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


## 1. Model initialization

We load previously trained titans weights and merge them with original Gemma weights

In [ ]:
!cp "/content/drive/Shareddrives/shared_veriga/saved_titans_delta/saved_titans_delta_distill_23.zip" saved_titans_delta.zip

In [ ]:
# !rm saved_titans_delta.zip

In [ ]:
import shutil
import os

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

merged_params = None
titans_delta_path = "./saved_titans_delta"
titans_zip = "saved_titans_delta.zip" #saved_titans_delta_short_context.zip
workdir_checkpoints = "./titans_workdir_squad/checkpoints"


if os.path.exists(workdir_checkpoints) and len(os.listdir(workdir_checkpoints)) > 0:
    print(f"📁 Найдена директория {workdir_checkpoints}. Пропускаем ручное слияние весов.")
    print("Kauldron автоматически загрузит последнее состояние при старте обучения.")
else:
    if os.path.exists(titans_zip):
        print(f"Unpacking {titans_zip}...")
        shutil.unpack_archive(titans_zip, titans_delta_path)

    if os.path.exists(titans_delta_path):
        # Load original Gemma 3 1B IT weights
        print("Loading original Gemma weights...")
        original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

        # Merge loaded Titans weights into original Gemma params
        print("Merging Titans weights...")
        import titans_tree_utils


        loaded_titans_params = load_titans_weights(titans_delta_path)
        merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params, remove_dead_attn=False)
        # # ###################################### удалить
        # embed_dim = 1152
        # key = jax.random.PRNGKey(42) # Фиксированный ключ для воспроизводимости

        # for layer_name in merged_params:
        #     if 'layer_' in layer_name and 'memory' in merged_params[layer_name]:
        #         if 'memory_gate_proj' not in merged_params[layer_name]:
        #             print(f"Properly initializing dynamic gate for {layer_name}")

        #             # Разделяем ключ для каждого слоя
        #             key, subkey = jax.random.split(key)

        #             # Используем стандартную инициализацию весов (Lecun Normal)
        #             # Она даст веса, которые в среднем дают 0 на выходе (sigmoid=0.5)
        #             new_kernel = jax.nn.initializers.lecun_normal()(
        #                 subkey, (embed_dim, embed_dim), jnp.bfloat16
        #             )

        #             merged_params[layer_name]['memory_gate_proj'] = {
        #                 'kernel': new_kernel
        #             }
        # # #####################################################
        print("✅ Success: Pre-trained Titans weights loaded and merged.")
    else:
        print("⚠️ Note: './saved_titans_delta' not found. Titans layers will be randomly initialized.")

## 2. Pre-training (CPT)

We use `Kauldron` to orchestrate the training.
- **Dataset:** Instead of a python generator, we use `kd.data.py.Elements`.
- **Model Config:** We use the standard `gm.nn.Gemma3_1B.config`.
- **SkipTitans:** Handles loading Gemma weights while keeping Titans randomized.
- **partial_updates:** Ensures XLA only builds backprop graphs for memory parameters to prevent OOM.

### hyper-parameters

In [ ]:
# Create a proper Kauldron dataset pipeline using TFDS
batch_size = 32
max_length = 2048 #
total_steps = 10000

titans_phase2_first_layer = 23  # Titans layers from this index onward are active.
                                 # Earlier layers revert to standard Gemma blocks.
                                 # 17 → layers 17,23 active (~25GB compile RAM)
                                 # 23 → layer 23 only  (~5GB compile RAM)

# Гиперпараметры Titans Memory
huber_loss_delta = optax.constant_schedule(0.1)
# optax.linear_schedule(
#    init_value=0.1, end_value=.5, transition_begin = 100, transition_steps=19900)
# )
# huber_loss_delta = optax.cosine_decay_schedule(
#     init_value=0.8,
#     decay_steps=10000,
#     alpha=0.1,
# )
  # Можно передать число или optax schedule

_all_titans_layers = (11, 17, 23)
active_titans_layers = tuple(l for l in _all_titans_layers if l >= titans_phase2_first_layer)
print(f"Active Titans layers: {active_titans_layers}")


In [ ]:
# !git pull

In [ ]:
# import importlib
# importlib.reload(gemma_titans)
# from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config


In [ ]:
# Define the model configuration
import dataclasses

gemma_config = dataclasses.replace(
    Gemma3_1B_Titans.config,
    titans_layer_indices=active_titans_layers,
    titans_phase2_first_layer=titans_phase2_first_layer,
    neural_mem_huber_delta=huber_loss_delta,
)

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config, # По умолчанию is_training_mode=True
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

def format_triviaqa(x):
    # В trivia_qa/rc структура: 'entity_pages' (list) или 'search_results'
    # Берем первый попавшийся контекст из entity_pages
    ctx = ""
    # if x['entity_pages']['wiki_context']:
    #     ctx = x['entity_pages']['wiki_context'][0]
    if x['search_results']['search_context']:
        ctx = x['search_results']['search_context'][0]
    q = x["question"]
    # Ответ в TriviaQA обычно в x['answer']['value']
    ans = x['answer']['value']

    x['src'] = f"Context: {ctx}\n\nAnswer the question in one sentence: {q}"
    x['dst'] = ans
    return x

tokenizer = gm.text.Gemma3Tokenizer()


In [ ]:
# import importlib
# # importlib.reload(adam_atan2)
# from adam_atan2 import adam_atan2

## dataset


### OpenWebText

In [ ]:
"""
Kauldron data loader module for OpenWebText (replaces c4/webtextlike).
"""

import os
import dataclasses
import numpy as np
import grain.python as grain
from kauldron import kd

tokenizer = gm.text.Gemma3Tokenizer()

@dataclasses.dataclass(frozen=True)
class CLMTask(grain.MapTransform):
    """Tokenize full document text → fixed-length tokens array for CLM training.

    Loss is computed on every token position — no src/dst masking.
    Suitable for Wikipedia, PG-19, C4, etc.
    """
    in_text: str
    out_tokens: str
    tokenizer: object
    max_length: int

    def map(self, element):
        text = element[self.in_text]
        if isinstance(text, bytes):
            text = text.decode("utf-8")
        token_ids = self.tokenizer.encode(text, add_bos=True)
        token_ids = token_ids[:self.max_length]
        pad_len = self.max_length - len(token_ids)
        import numpy as np
        token_ids = np.pad(np.array(token_ids, dtype=np.int32), (0, pad_len))
        out = dict(element)
        out[self.out_tokens] = token_ids
        return out

@dataclasses.dataclass(frozen=True)
class TokenizeText(grain.MapTransform):
    """Tokenizes the text field into a fixed-length tokens array."""
    in_text: str = "text"
    out_tokens: str = "tokens"
    tokenizer: object = None
    max_length: int = 1024

    def map(self, element: dict) -> dict:
        # HuggingFace datasets usually provide strings directly, no decoding needed
        text = element[self.in_text]

        # Assuming tokenizer has an encode method (like sentencepiece)
        if hasattr(self.tokenizer, "encode"):
            # Check if add_bos is supported, some tokenizers use it
            try:
                token_ids = self.tokenizer.encode(text, add_bos=True)
            except TypeError:
                token_ids = self.tokenizer.encode(text)
        else:
            token_ids = self.tokenizer(text)

        token_ids = token_ids[:self.max_length]
        pad_len = self.max_length - len(token_ids)
        token_ids = np.pad(np.array(token_ids, dtype=np.int32), (0, pad_len))

        out = dict(element)
        out[self.out_tokens] = token_ids
        return out


def get_c4_webtextlike_dataset(
    split: str = "train",
    batch_size: int = 8,
    tokenizer: object = None,
    max_length: int = 1024,
    data_dir: str = None, # Unused for HuggingFace, kept for API compatibility
    num_epochs: int = None,
    shuffle: bool = True,
):
    """
    Returns a Kauldron dataset pipeline for OpenWebText via HuggingFace.
    This replaces the broken c4/webtextlike TFDS builder.

    Args:
        split: The dataset split to load (e.g., 'train', 'validation').
        batch_size: Number of examples per batch.
        tokenizer: Optional tokenizer object to tokenize the text.
                   If None, raw text is returned.
        max_length: Maximum sequence length for tokenization.
        data_dir: Unused. Kept for API compatibility.
        num_epochs: Number of epochs to iterate (None for infinite).
        shuffle: Whether to shuffle the dataset.

    Returns:
        A Kauldron data source iterator.
    """
    transforms = []

    if tokenizer is not None:
        transforms.append(
            CLMTask(
                in_text="text",
                out_tokens="tokens",
                tokenizer=tokenizer,
                max_length=max_length
            )
        )
        keep_fields = ["tokens"]
    else:
        keep_fields = ["text"]

    transforms.append(kd.data.py.Elements(keep=keep_fields))

    # Using the Kauldron HuggingFace loader to completely bypass Apache Beam
    dataset = kd.data.py.HuggingFace(
        path="Skylion007/openwebtext",
        num_workers=1,
        split=split,
        shuffle=shuffle,
        num_epochs=num_epochs,
        batch_size=batch_size,
        shard_by_process=False,
        transforms=transforms,
        cache_dir='/content/drive/Shareddrives/shared_veriga/tfds_cache',
    )

    return dataset


### trivia_qa/rc

In [ ]:
# 1. Выбираем TyDi QA
import grain
MAX_CONTEXT_CHARS = (max_length - 50) * 4  # ~3700 символов для max_length=1024

class FilterShortContext(grain.python.FilterTransform):
    def filter(self, x):
        ctxs = x['search_results']['search_context']
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_train_dataset_tydi_qa():
    return kd.data.py.Tfds(
        name= "trivia_qa/rc", #splits: 'test'	17,210 | 'train'	138,384 | 'validation'	18,669
        split="train",
        shuffle=True,
        num_epochs=None,
        batch_size=batch_size,
        num_workers=1,
        transforms=[
            # поля context/question/answers
            FilterShortContext(),
            format_triviaqa, # format_squad,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="tokens",
                out_target="target",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["tokens", "target"]),
        ],
    )




#### загрузка отфильтрованного датасета с короткими контекстами

In [ ]:
os.path.abspath('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')

In [ ]:
import pickle

DATA_DIR = os.path.abspath('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')

def get_precomputed_dataset(batch_size=16, tokenizer=None, max_length=1024, files=None):
    """Загружает и объединяет отфильтрованные данные из нескольких файлов.

    Args:
        batch_size: Размер батча.
        tokenizer: Токенизатор Gemma.
        max_length: Максимальная длина последовательности.
        files: Список имен файлов, например ['train.array_record', 'validation.array_record']
    """
    if files is None:
        files = ['train_gemma_generated.array_record']

    paths = [os.path.join(DATA_DIR, f) for f in files]
    print(f"Загрузка данных из: {paths}")

    # 1. Определяем источник данных (DataSource)
    class PickledArrayRecordDataSource(grain.python.ArrayRecordDataSource):
        def __getitem__(self, idx):
            return pickle.loads(super().__getitem__(idx))

    data_source = PickledArrayRecordDataSource(paths)

    # 2. Создаем Kauldron Dataset
    return kd.data.py.DataSource(
        seed=42,
        data_source=data_source,
        shuffle=True,      # Перемешивание важно при объединении разных сплитов!
        num_epochs=None,   # Бесконечно для обучения
        batch_size=batch_size,
        transforms=[
            format_triviaqa, # format_squad,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="tokens",
                out_target="target",
                out_target_mask="loss_mask",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["tokens", "target"]),
        ],
    )

In [ ]:
print(dir(grain.python))

### optimizer routing

In [ ]:
from adam_atan2 import adam_atan2
from optax.contrib._muon import MuonDimensionNumbers

def muon_only_dims(params):
    MUON_KEYS = {'to_queries', 'to_keys_values', 'combine_heads'}
    def _label(path, v):
        key    = str(path[-1].key) if hasattr(path[-1], 'key') else ''
        parent = str(path[-2].key) if len(path) > 1 and hasattr(path[-2], 'key') else ''
        if key == 'kernel' and parent in MUON_KEYS:
            return MuonDimensionNumbers(reduction_axis=0, output_axis=1)
        return None
    return jax.tree_util.tree_map_with_path(_label, params)

def is_muon_param(path_str):
    parts = path_str.split('/')
    return (len(parts) >= 2 and
            parts[-1] == 'kernel' and
            parts[-2] in {'to_queries', 'to_keys_values', 'combine_heads'})

def muon_mask(params):
    def _m(path, v):
        return is_muon_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def is_gate_param(path_str):
    # Проверяем, относится ли параметр к гейту
    return 'memory_gate_proj' in path_str.split('/')

def gate_mask(params):
    def _m(path, v):
        return is_gate_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def adam_base_mask(params):
    def _m(path, v):
        path_str = '/'.join(str(p.key) for p in path)
        return not is_muon_param(path_str) and not is_gate_param(path_str)
    return jax.tree_util.tree_map_with_path(_m, params)



# Сниженные LR: веса уже предобучены в Phase 1
lr_muon = optax.warmup_cosine_decay_schedule(
    init_value=1e-4,
    peak_value=5e-4,
    warmup_steps=500,
    decay_steps=total_steps - 500,
    end_value=1e-5,
)
# lr_muon = optax.cosine_decay_schedule(
#     init_value=1e-5,
#     decay_steps=total_steps,
#     alpha=0.05,  # конечный lr = 1e-3 * 0.05 = 5e-5
# )
lr_adam = optax.warmup_cosine_decay_schedule(
    init_value=1e-4,
    peak_value=1e-4,
    warmup_steps=500,
    decay_steps=total_steps - 500,
    end_value=1e-5,
)

lr_gate = optax.constant_schedule(0.1)
# warmup_cosine_decay_schedule(
#     init_value=1e-2,
#     peak_value=5e-2,
#     warmup_steps=500,
#     decay_steps=total_steps - 500,
#     end_value=5e-3,
# )

## lr for experiments 4e-4, 1.5e-4
inner_chain = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.masked(
        optax.contrib.muon(
            learning_rate=lr_muon,
            muon_weight_dimension_numbers=muon_only_dims,
            # beta=0.80,
            eps=1e-8,
            mu_dtype=jnp.float32,
        ),
        mask=muon_mask,
    ),
    # 2. Adam для гейтов (с высоким LR)
    optax.masked(
        adam_atan2(
            learning_rate=lr_gate,
            b1=0.90, b2=0.90,
            eps=1e-8,
            mu_dtype=jnp.float32
        ),
        mask=gate_mask,
    ),
    # 3. Adam для остальных параметров (с обычным LR)
    optax.masked(
        adam_atan2(
            learning_rate=lr_adam,
            b1=0.90,
            b2=0.80,
            eps=1e-8,
            mu_dtype=jnp.float32
        ),
        mask=adam_base_mask,
    ),
)

routing_optimizer = optax.MultiSteps(
    kd.optim.partial_updates(
        inner_chain,
        mask=kd.optim.select(["memory", "memory_gate_proj"]),
    ),
    every_k_schedule=16,
)

In [ ]:
step = 500
print(f"lr_gate {lr_gate(step)}")
print(f"lr_muon {lr_muon(step)}")
print(f"lr_adam {lr_adam(step)}")
print(f"huber {huber_loss_delta(step)}")

### metrics

In [ ]:
import flax
import flax.linen as nn
from kauldron import metrics as kd_metrics
from kauldron import kontext

class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)

# LM loss — основной сигнал Phase 2
train_losses = {
    "lm_loss": kd.losses.Value(
        values="preds.layer_losses['lm_loss']"
    )
}

# Метрики
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class MemoryGateMetric(kd_metrics.Metric):
    params: kd.kontext.Key = "params"

    @flax.struct.dataclass
    class State(kd_metrics.State):
        gate_metrics: flax.core.FrozenDict[str, jnp.ndarray] = flax.core.FrozenDict()
        def compute(self):
            return {k: np.array(v, dtype=np.float32) for k, v in self.gate_metrics.items()}
        @classmethod
        def empty(cls):
            return cls(gate_metrics=flax.core.FrozenDict())
        def merge(self, other):
            return self

    def get_state(self, params=None, **kwargs) -> State:
        if params is None:
            return self.State.empty()
        stats_dict = {}
        def find_gates(tree, stats_dict: dict, path_prefix: str = ""):
            """
            Рекурсивно ищет 'memory_gate_proj' в дереве параметров и заполняет stats_dict метриками.
            """
            for key, val in tree.items():
                current_path = f"{path_prefix}_{key}" if path_prefix else str(key)
                if key == "memory_gate_proj":
                    # Handle Dense layer parameters (dict or single array)
                    leaves = jax.tree_util.tree_leaves(val)
                    all_params = jnp.concatenate([jnp.ravel(p) for p in leaves])
                    mean_val = jnp.mean(all_params)
                    openness = jax.nn.sigmoid(mean_val)
                    stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                    stats_dict[f"titans_gates/{current_path}_openness"] = openness
                    std_val = jnp.std(all_params)
                    stats_dict[f"titans_gates/{current_path}_raw_std"] = std_val
                    # Извлекаем веса ядра (kernel) из Dense-слоя
                    if isinstance(val, (dict, flax.core.FrozenDict)):
                        actual_weights = val.get('kernel', val)
                    else:
                        actual_weights = val

                    mean_val = jnp.mean(actual_weights)
                    openness = jax.nn.sigmoid(mean_val)
                    stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                    stats_dict[f"titans_gates/{current_path}_openness"] = openness
                    stats_dict[f"titans_gates/{current_path}_raw_std"] = jnp.std(actual_weights)
                elif isinstance(val, (dict, flax.core.FrozenDict)):
                    # Продолжаем поиск в поддереве, передавая словарь для накопления статистики
                    find_gates(val, stats_dict, current_path)

        return self.State(gate_metrics=flax.core.freeze(stats_dict))

@flax.struct.dataclass
class LRState(kd_metrics.State):
    lr_value: jnp.ndarray
    @classmethod
    def empty(cls):
        return cls(lr_value=jnp.array(0.0))
    def merge(self, other):
        return self
    def compute(self):
        return self.lr_value

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class HuberDeltaMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(huber_loss_delta(step)))
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class AdamLearningRateMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(lr_adam(step)))

class TPUMemoryMetric(kd_metrics.Metric):
    """Метрика для логирования использования памяти TPU в ГБ."""
    @flax.struct.dataclass
    class State(kd_metrics.State):
        # Состояние может быть пустым, так как данные мы берем напрямую из JAX на хосте
        def compute(self):
            stats_dict = {}
            for i, device in enumerate(jax.devices()):
                try:
                    m_stats = device.memory_stats()
                    # Если словарь пустой, пропускаем устройство
                    if not m_stats:
                        continue
                    prefix = f"device_{i}"
                    # 1. Текущее использование памяти (если ключа нет, вернет 0)
                    if 'bytes_in_use' in m_stats:
                        used_gb = m_stats['bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/used_gb"] = np.array(used_gb, dtype=np.float32)
                    # 2. Пиковое использование
                    if 'peak_bytes_in_use' in m_stats:
                        peak_gb = m_stats['peak_bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/peak_gb"] = np.array(peak_gb, dtype=np.float32)
                    # 3. Лимит и процент (только если 'limit' действительно существует)
                    if 'limit' in m_stats and 'bytes_in_use' in m_stats:
                        limit_gb = m_stats['limit'] / 1e9
                        usage_pct = (m_stats['bytes_in_use'] / m_stats['limit']) * 100
                        stats_dict[f"{prefix}/usage_pct"] = np.array(usage_pct, dtype=np.float32)
                except (AttributeError, ValueError, RuntimeError):
                    pass
            return stats_dict
        @classmethod
        def empty(cls):
            """Создает пустое начальное состояние."""
            return cls()
        def merge(self, other):
            """Объединяет состояния из разных батчей (здесь ничего не делаем)."""
            return self

    def get_state(self, **kwargs) -> State:
        # Просто возвращаем пустое состояние.
        # Нам не нужны данные из batch или модели для этой метрики.
        return self.State().empty()

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class Perplexity(kd.metrics.Metric):
    # Берет усредненный loss по батчу
    loss: kd.kontext.Key = "preds.layer_losses['lm_loss']"

    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        def compute(self):
            # Средний loss за все шаги -> возводим в экспоненту
            mean_loss = super().compute()
            return jnp.exp(mean_loss)
    def get_state(self,*, loss) -> State:
        return self.State.from_values(values=loss)

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class LMAccuracy(kd.metrics.Metric):
    # Берет точность по батчу
    acc: kd.kontext.Key = "preds.layer_losses['lm_accuracy']"
    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        pass
    def get_state(self,*, acc) -> State:
        return self.State.from_values(values=acc)


train_metrics = {
    # "LM/accuracy": LMAccuracy(),
    # "LM/perplexity": Perplexity(),
    "LM/adam_lr": AdamLearningRateMetric(),
    "memory_gate_proj": MemoryGateMetric(),
    "tpu_memory": TPUMemoryMetric()
}
train_summaries = {}
for layer in active_titans_layers:
    key = f"Gates_Dist_{layer}"
    train_summaries[key] = kd.summaries.HistogramSummary(
        tensor=f"params.layer_{layer}.memory_gate_proj.kernel"
        )


In [ ]:
import flax.linen as nn
import grain
# Configure Trainer


trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath('./titans_workdir_squad'),
    train_ds = get_c4_webtextlike_dataset(
        split="train",
        batch_size=batch_size,
        tokenizer=gm.text.Gemma3Tokenizer(), # pass tokenizer instance here
        shuffle=True,
        num_epochs=None
    ),
    # get_precomputed_dataset(
    #     batch_size=batch_size,
    #     tokenizer=tokenizer,
    #     max_length=max_length,
    #     files=[
    #         # 'validation_gemma_generated.array_record',
    #         # 'test_gemma_generated.array_record',
    #         'train_gemma_generated.array_record'
    #         ]
    # ),
    model= model,
    # If we have merged_params from Cell 1, use them directly (much faster).
    # Otherwise, use SkipTitans to load Gemma and randomly init Titans.
    init_transform = FullParamsInit(merged_params) if ('merged_params' in dir()) and (merged_params is not None)  else SkipTitans(
        wrapped=gm.ckpts.LoadCheckpoint(
            path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
        ),
        workdir=os.path.abspath('./SkipTitans_workdir'),
    ),
    checkpointer=kd.ckpts.Checkpointer(
        save_interval_steps=500,
    ),

    num_train_steps = total_steps, #87600 // 16, #dataset length times to epoches num divide to batch_size
    train_losses=train_losses,
    train_metrics=train_metrics,
    train_summaries = train_summaries,
    optimizer = routing_optimizer,

    # checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),

    # Sharding for TPU / TPU Pods
    # sharding=kd.sharding.ShardingStrategy(),
    # train_metrics={

    #     # Наш монитор памяти
    #     "tpu_memory": TPUMemoryMetric()
    # },
)

print("Trainer initialized. Starting compilation and training...")


### 2.5 Monitoring with TensorBoard
Launch TensorBoard to monitor training metrics (Loss, Learning Rate) in real-time.

In [ ]:
%reload_ext tensorboard


In [ ]:
from tensorboard import notebook

# Показать список всех активных инстансов
notebook.list()

In [ ]:
!rm -rf /tmp/.tensorboard-info/*

In [ ]:
%tensorboard --logdir ./titans_workdir_squad/ --port=6007

## Start training

In [ ]:
# run the actual training loop:
state, aux = trainer.train()

In [ ]:
import orbax.checkpoint as ocp
import jax.numpy as jnp
import os
import titans_tree_utils

#   1. Находим последний чекпоинт
workdir = "/content/Titans_jax/titans_workdir_squad"
ckpt_root = os.path.join(workdir, "checkpoints")
all_steps = [int(d.split('_')[-1]) for d in os.listdir(ckpt_root) if d.startswith('ckpt_')]
latest_step = max(all_steps)
step_path = os.path.join(ckpt_root, f"ckpt_{latest_step}")

#   2. Ищем, где именно лежат данные (подпапка с файлом _METADATA)
data_path = None
for root, dirs, files in os.walk(step_path):
    if '_METADATA' in files:
        data_path = root
        break

if not data_path:
    raise FileNotFoundError(f"Не удалось найти данные в {step_path}")

print(f"Загружаю обученные за 1.5 часа веса из: {data_path}")

#   3. Загружаем и извлекаем параметры
checkpointer = ocp.StandardCheckpointer()
saved_state = checkpointer.restore(data_path)

# В Kauldron параметры обычно лежат в корне или в ключе 'params'
if 'params' in saved_state:
    saved_params = saved_state['params']
else:
    saved_params = saved_state

#   4. Собираем итоговое дерево
#   (Убедитесь, что original_params и loaded_titans_params загружены ранее)
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

# Merge loaded Titans weights into original Gemma params
print("Merging Titans weights...")
import titans_tree_utils

if os.path.exists(titans_zip):
    print(f"Unpacking {titans_zip}...")
    shutil.unpack_archive(titans_zip, titans_delta_path)
loaded_titans_params = load_titans_weights(titans_delta_path)

merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

#   5. Переносим обученные веса гейта
found_any = False
for layer_name in merged_params:
    if 'layer_' in layer_name:
        #   Проверяем наличие гейта в сохраненных параметрах
        if 'memory_gate_proj' in saved_params.get(layer_name, {}):
            print(f"✅ Восстановлен обученный гейт для {layer_name}")
            merged_params[layer_name]['memory_gate_proj'] = saved_params[layer_name]['memory_gate_proj']
            found_any = True

if not found_any:
      print("❌ ПРЕДУПРЕЖДЕНИЕ: В чекпоинте не найдены веса memory_gate_proj!")

In [ ]:
import jax
import gc

# 1. DESTROY the Python references to the old TPU buffers FIRST
try:
    del state
    del aux
except NameError:
    pass

# 2. Clear JAX's internal live buffer cache
for device in jax.devices():
    if hasattr(device, 'live_arrays'): device.live_arrays().clear()
    if hasattr(device, 'live_buffers'): device.live_buffers().clear()
    if hasattr(device, 'default_memory_tracker'): device.default_memory_tracker().clear()

jax.clear_caches()

# 3. NOW run Python garbage collection
gc.collect()

print("TPU memory cache cleared and garbage collection finished.")


In [ ]:

trainer = trainer.replace(num_train_steps=20000)

state, aux = trainer.train()


In [ ]:
import os
# os._exit(0)

## 3. Save / Load Trained Weights
We don't want to save the entire 1B model, just our new memory projections. We utilize the `split_titans_params` utility.

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    # 1. Extract only the Titans weights
    _, titans_params = titans_tree_utils.split_titans_params(state.params)

    import shutil
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)

    checkpointer = ocp.StandardCheckpointer()

    # 2. Correct Orbax syntax: save(path, state)
    # Passing titans_params as the positional state saves only them.
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved ONLY Titans weights to {save_path}")


# Usage:
save_titans_weights(state, "./saved_titans_delta")
# loaded_titans_params = load_titans_weights("./saved_titans_delta")
# original_params, _ = titans_tree_utils.split_titans_params(state.params)
# state = state.replace(params=titans_tree_utils.merge_titans_params(original_params, loaded_titans_params))



In [ ]:
!ls ./saved_titans_delta


In [ ]:
import os
import zipfile
import shutil
from google.colab import files

checkpoint_dir = "./saved_titans_delta"

# First, let's check if the directory exists
if not os.path.exists(checkpoint_dir):
    print(f"Error: {checkpoint_dir} does not exist!")
else:
    # Try Colab download first
    try:

        # Create a zip file of the checkpoint
        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)
        dest_file = f"/content/drive/Shareddrives/shared_veriga/saved_titans_delta/saved_titans_delta_distill_{titans_phase2_first_layer}.zip"
        !cp ./saved_titans_delta.zip {dest_file}

        # Download the zip file
        files.download(zip_filename)
        print(f"✓ Downloaded {zip_filename} via Colab")

    except ImportError:
        # Not in Colab, create zip and provide manual download instructions
        print("Not running in Google Colab. Creating zip file...")

        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)

        # Get file size
        file_size = os.path.getsize(zip_filename) / (1024 * 1024)  # Size in MB

        print(f"\n✓ Created {zip_filename} ({file_size:.2f} MB)")
        print(f"📁 Full path: {os.path.abspath(zip_filename)}")
        print("\nTo download, use one of these methods:")
        print("1. Right-click the file in Jupyter file browser and select 'Download'")
        print("2. Use scp/rsync from your terminal:")
        print(f"   scp user@server:{os.path.abspath(zip_filename)} ./")